# 05 - Project Summary

A short close-out: what this project found, what it doesn't tell us, and
what would come next. Details, figures, and numbers behind every point here
are in `notebooks/01-04` — this notebook doesn't repeat them, only summarizes.

## Main findings

- **Transfer learning wins decisively.** EfficientNetB0 (fine-tuned) reached
  95.8% test accuracy vs. 60.0% for a CNN trained from scratch on the same
  ~6,551 training images — a 35.8-point gap on a consistent comparison using the same data split
  and evaluation protocol (same preprocessing, same augmentation).
- **Fine-tuning earns its keep.** Fine-tuning the backbone gave a modest
  but consistent gain over the frozen backbone (94.4% -> 95.8%). Unfreezing
  a small part of the backbone at a low learning rate improves adaptation to
  the flower dataset without substantially disrupting the pretrained
  representation.
- **The model knows what it doesn't know.** Well-calibrated (Expected
  Calibration Error 0.020) with a large confidence gap between correct
  (95.4%) and incorrect (63.7%) predictions — promising enough to support a
  prototype abstention rule (chosen on validation data, confirmed once on
  test — never tuned on test), although further validation would be
  required before deployment.
- **Mistakes are explainable, not random.** Errors concentrate on a small
  number of visually similar species pairs, and top-3 accuracy (99.0%) is
  far higher than top-1 (95.8%) — most misses are the right species ranked
  2nd or 3rd, not an arbitrary guess.
- **The split is verified, not just declared.** Reproducible on rerun with
  the same seed, and the train/validation/test index sets are provably
  disjoint by construction (`tests/test_data.py`).

## Limitations

- **Not benchmark-comparable.** This project uses a custom stratified
  80/10/10 split, not the dataset's official split, to get a realistic amount
  of training data and real class imbalance to analyze. Reported accuracy
  should not be quoted alongside official Oxford Flowers 102 leaderboard
  numbers.
- **Small test set per class.** 819 test images across 102 classes averages
  ~8 images per species — per-class F1 and confusion-pair counts are
  indicative, not statistically precise, for any single species.
- **No duplicate-image audit.** The train, validation and test index sets
  are disjoint, but exact and near-duplicate image detection across splits
  was outside the scope of this project.
- **Closed-set only.** The model always picks one of its 102 trained
  species; it has no built-in way to say "none of these" for an
  out-of-distribution flower.
- **Clean training data, unknown real-world performance.** Training photos
  are relatively centered, well-lit, single-flower shots. Accuracy on messy
  real user photos (poor lighting, multiple flowers, occlusion, unfamiliar
  species) is unvalidated.
- **Imbalance affects reliability unevenly.** Rarer species (as few as 32
  training images) are learned from less evidence and are more likely to be
  misclassified than well-represented ones.

## Future work

- **Broader, noisier data.** Collect or augment with photos closer to real
  deployment conditions, not just curated benchmark images.
- **Out-of-distribution detection.** A dedicated "unknown species" mechanism,
  beyond the current fixed confidence threshold in `app/app.py`.
- **Production monitoring.** Track the live confidence distribution over
  time as an early signal of data drift, per `docs/business_case.md`.
- **Platform validation.** Benchmark latency and footprint on an actual
  target device (mobile vs. server) before any real deployment decision.
- **Official-split benchmark run.** For anyone who needs a number directly
  comparable to published Oxford Flowers 102 results, rerun the same
  baseline-vs-transfer-learning comparison on the dataset's official split
  as a separate, clearly labeled experiment.